# Модуль 1 · Урок 7. Pandas: groupby, pivot_table, merge
**AI-Driven ML Engineer · Data Science Academy**

| Время | Блок |
|---|---|
| 0–10 | Разминка «что выведет код» |
| 10–25 | Разбор ДЗ 5 и ДЗ 6 (на слайдах) · устная проверка |
| 25–60 | groupby и agg: «по каждому городу» одной строкой · Практика 1 |
| 60–95 | pivot_table: город × месяц, что изменилось в Шымкенте · Практика 2 |
| 95–110 | merge: заказы + клиенты, проверяем гипотезу маркетолога · Практика 3 |
| 110–120 | Итоги, ДЗ 7 |

**Правило занятия:** предскажи → запусти → объясни. **Правило среза:** рядом с каждой долей или средним показываем, на скольких заказах оно посчитано.

## Разминка: что выведет код? (30 секунд на вопрос, без запуска)
```python
import numpy as np
import pandas as pd
# 1
tot = {}
for c, a in [("A", 10), ("B", 5), ("A", 7)]:
    tot[c] = tot.get(c, 0) + a
print(tot)
# 2
s = pd.Series(["оплачен", "отменён", "оплачен", "оплачен"])
print((s == "отменён").mean())
# 3
print(pd.Series([10, 20, 30], index=["a", "b", "a"])["a"].sum())
# 4
x = pd.Series([1, 2], index=["x", "y"])
y = pd.Series([10, 20], index=["y", "z"])
print((x + y).tolist())
# 5
a = [100]
b = [300, 300, 300]
print((np.mean(a) + np.mean(b)) / 2, np.mean(a + b))
```

In [1]:
import numpy as np
import pandas as pd

In [2]:
tot = {}
for c, a in [("A", 10), ("B", 5), ("A", 7)]:
    tot[c] = tot.get(c, 0) + a
    print(tot)
print(tot)

{'A': 10}
{'A': 10, 'B': 5}
{'A': 17, 'B': 5}
{'A': 17, 'B': 5}


In [3]:
s = pd.Series(["оплачен", "отменён", "оплачен", "оплачен"])


In [4]:
s == "отменён"

0    False
1     True
2    False
3    False
dtype: bool

In [5]:
print((s == "отменён").mean())

0.25


In [6]:
pd.Series([10, 20, 30], index=["a", "b", "a"])["a"]

a    10
a    30
dtype: int64

In [7]:
pd.Series([10, 20, 30], index=["a", "b", "a"])["a"].sum()

np.int64(40)

In [8]:
x = pd.Series([1, 2], index=["x", "y"])
y = pd.Series([10, 20], index=["y", "z"])


In [9]:
x

x    1
y    2
dtype: int64

In [10]:
y

y    10
z    20
dtype: int64

In [11]:
x + y

x     NaN
y    12.0
z     NaN
dtype: float64

In [12]:
(x + y).tolist()

[nan, 12.0, nan]

In [13]:
a = [100]
b = [300, 300, 300]


In [14]:
np.mean(a)

np.float64(100.0)

In [15]:
np.mean(b)

np.float64(300.0)

In [16]:
(np.mean(a) + np.mean(b)) / 2

np.float64(200.0)

In [17]:
a + b

[100, 300, 300, 300]

In [18]:
np.mean(a + b)

np.float64(250.0)

In [19]:
print((np.mean(a) + np.mean(b)) / 2, np.mean(a + b))

200.0 250.0


## Данные: orders.csv после очистки урока 6 и customers.csv
Очистку урока 6 собрали в одну ячейку. Проверка: 5 280 строк.

In [20]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [21]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [22]:
df["cancel"] = df["status"] == "отменён"     # True/False: пригодится для доли отмен
paid = df[df["status"] == "оплачен"]
len(paid)

4739

In [23]:
df

,order_id,order_date,customer_id,city,category,amount,delivery_days,status,month,cancel
0,101674,2026-09-01 08:00:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен,9,False
1,101214,2026-09-01 08:00:00,C01210,Астана,Красота,3630.0,2.0,оплачен,9,False
2,102424,2026-09-01 08:00:00,C01408,Караганда,Дом,9120.0,3.0,оплачен,9,False
3,102333,2026-09-01 08:00:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен,9,False
4,102564,2026-09-01 09:00:00,C00657,Астана,Дом,11010.0,1.0,оплачен,9,False
...,...,...,...,...,...,...,...,...,...,...
5458,104168,2026-10-31 21:00:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен,10,False
5459,104648,2026-10-31 21:00:00,C00629,Караганда,Книги,3640.0,2.0,оплачен,10,False
5460,104615,2026-10-31 22:00:00,C00557,Астана,Электроника,22150.0,NaN,оплачен,10,False
5461,103460,2026-10-31 22:00:00,C02162,Алматы,Электроника,24670.0,2.0,отменён,10,True


## 1. groupby: «по каждому X» одной строкой (05-Groupby)
На уроке 2 мы считали «по каждому городу» словарём-накопителем (вопрос 1 разминки). `groupby` делает то же самое в три шага: **разбить** строки на группы по значению столбца → **посчитать** в каждой группе → **собрать** результат в Series с ярлыками групп.

In [24]:
toy = pd.DataFrame({"city": ["Алматы", "Шымкент", "Алматы", "Шымкент", "Астана", "Алматы"],
                    "amount": [100, 200, 300, 400, 500, 600],
                    "status": ["оплачен", "отменён", "оплачен", "оплачен", "оплачен", "отменён"]})
toy

,city,amount,status
0,Алматы,100,оплачен
1,Шымкент,200,отменён
2,Алматы,300,оплачен
3,Шымкент,400,оплачен
4,Астана,500,оплачен
5,Алматы,600,отменён


In [25]:
toy.groupby("city")      # сам по себе groupby ничего не считает

In [26]:
toy.groupby("city")["amount"].sum()

city
Алматы     1000
Астана      500
Шымкент     600
Name: amount, dtype: int64

In [27]:
# ПРЕДСКАЖИТЕ: среднее и число строк в каждой группе
toy.groupby("city")["amount"].mean()

city
Алматы     333.333333
Астана     500.000000
Шымкент    300.000000
Name: amount, dtype: float64

In [28]:
toy.groupby("city").size()

city
Алматы     3
Астана     1
Шымкент    2
dtype: int64

In [29]:
toy.groupby("city")["amount"].agg(["sum", "mean", "count"])

,sum,mean,count
city,,,
Алматы,1000,333.333333,3
Астана,500,500.000000,1
Шымкент,600,300.000000,2


In [30]:
toy.describe().T

,count,mean,std,min,25%,50%,75%,max
amount,6.0,350.0,187.082869,100.0,225.0,350.0,475.0,600.0


In [31]:
toy.groupby("city").describe()

amount                                                           
         count        mean         std    min    25%    50%    75%    max
city                                                                     
Алматы     3.0  333.333333  251.661148  100.0  200.0  300.0  450.0  600.0
Астана     1.0  500.000000         NaN  500.0  500.0  500.0  500.0  500.0
Шымкент    2.0  300.000000  141.421356  200.0  250.0  300.0  350.0  400.0

Результат – Series или таблица, где **ярлыки строк – названия групп**. Дальше работает всё, что мы знаем: `sort_values`, `loc`, арифметика с выравниванием по ярлыкам.

### groupby на orders.csv

In [32]:
df['city'].value_counts()

city
Алматы       2116
Астана       1339
Шымкент       776
Караганда     519
Актобе        387
Атырау        143
Name: count, dtype: int64

In [33]:
df.groupby("city").size()          # то же, что value_counts, но по алфавиту

city
Актобе        387
Алматы       2116
Астана       1339
Атырау        143
Караганда     519
Шымкент       776
dtype: int64

In [34]:
# Выручка: только оплаченные заказы
paid.groupby("city")["amount"].sum()

city
Актобе        5117380.0
Алматы       29144090.0
Астана       17884750.0
Атырау        1893850.0
Караганда     7020200.0
Шымкент       8602660.0
Name: amount, dtype: float64

In [35]:
df.groupby("city")["amount"].sum()

city
Актобе        5591740.0
Алматы       31578810.0
Астана       19487630.0
Атырау        2094710.0
Караганда     7742960.0
Шымкент      10945960.0
Name: amount, dtype: float64

> **Частая ошибка в коде от AI:** `df.groupby("city")["amount"].sum()` без фильтра статуса. В «выручку» попадают отменённые заказы, а их в Шымкенте каждый пятый.

### Доля отмен: среднее от True/False (вопрос 2 разминки)

In [36]:
df.groupby("city")["cancel"].mean()

city
Актобе       0.105943
Алматы       0.077505
Астана       0.088125
Атырау       0.076923
Караганда    0.090559
Шымкент      0.206186
Name: cancel, dtype: float64

In [37]:
df.groupby("city")["cancel"].mean().sort_values().round(3)

city
Атырау       0.077
Алматы       0.078
Астана       0.088
Караганда    0.091
Актобе       0.106
Шымкент      0.206
Name: cancel, dtype: float64

Доля без числа заказов вводит в заблуждение: в ДЗ 5 у написания «шымкент» было 3,7% отмен на 54 заказах. Поэтому считаем несколько показателей сразу и называем столбцы: `agg(имя=("столбец", "функция"))`.

In [38]:
df.groupby("city").agg(orders=("order_id", "count"))

,orders
city,
Актобе,387
Алматы,2116
Астана,1339
Атырау,143
Караганда,519
Шымкент,776


In [39]:
df.groupby("city").agg(cancel_share=("cancel", "mean"))

,cancel_share
city,
Актобе,0.105943
Алматы,0.077505
Астана,0.088125
Атырау,0.076923
Караганда,0.090559
Шымкент,0.206186


In [40]:
by_city = df.groupby("city").agg(orders=("order_id", "count"),
                                 cancel_share=("cancel", "mean"),
                                 avg_days=("delivery_days", "mean"))
by_city.sort_values("cancel_share").round(3)

,orders,cancel_share,avg_days
city,,,
Атырау,143,0.077,2.036
Алматы,2116,0.078,1.997
Астана,1339,0.088,2.019
Караганда,519,0.091,1.941
Актобе,387,0.106,2.070
Шымкент,776,0.206,4.451


In [41]:
df["delivery_days"].value_counts()

delivery_days
3.0    1590
2.0    1570
1.0    1557
6.0     120
8.0     104
7.0      95
5.0      86
Name: count, dtype: int64

In [42]:
df["delivery_days"].unique()

array([ 1.,  2.,  3., nan,  5.,  6.,  7.,  8.])

In [43]:
df["delivery_days"].isna().sum()

np.int64(158)

In [44]:
df.groupby("city").size()

city
Актобе        387
Алматы       2116
Астана       1339
Атырау        143
Караганда     519
Шымкент       776
dtype: int64

In [45]:
df.groupby("city")["delivery_days"].count()

city
Актобе        383
Алматы       2053
Астана       1299
Атырау        138
Караганда     495
Шымкент       754
Name: delivery_days, dtype: int64

In [46]:
# ПРЕДСКАЖИТЕ: одинаковые ли числа? Вспомните 160 пропусков в delivery_days
df.groupby("city")["delivery_days"].count() - df.groupby("city").size()

city
Актобе       -4
Алматы      -63
Астана      -40
Атырау       -5
Караганда   -24
Шымкент     -22
dtype: int64

`count` считает **непустые** значения столбца, `size` – все строки группы. Разница – пропуски.

### Две группировки сразу: город и месяц

In [47]:
df.groupby(["city", "month"])[["cancel"]].mean()

cancel
city      month          
Актобе    9      0.105263
          10     0.106599
Алматы    9      0.080537
          10     0.074557
Астана    9      0.093233
          10     0.083086
Атырау    9      0.086957
          10     0.067568
Караганда 9      0.087591
          10     0.093878
Шымкент   9      0.089136
          10     0.306954

In [48]:
df.groupby(["month", "city"])[["cancel"]].mean()

cancel
month city               
9     Актобе     0.105263
      Алматы     0.080537
      Астана     0.093233
      Атырау     0.086957
      Караганда  0.087591
      Шымкент    0.089136
10    Актобе     0.106599
      Алматы     0.074557
      Астана     0.083086
      Атырау     0.067568
      Караганда  0.093878
      Шымкент    0.306954

In [49]:
df.groupby(["city", "month"])["cancel"].mean()

city       month
Актобе     9        0.105263
           10       0.106599
Алматы     9        0.080537
           10       0.074557
Астана     9        0.093233
           10       0.083086
Атырау     9        0.086957
           10       0.067568
Караганда  9        0.087591
           10       0.093878
Шымкент    9        0.089136
           10       0.306954
Name: cancel, dtype: float64

In [50]:
df.groupby(["city", "month"])["cancel"].mean().unstack()

month,9,10
city,,
Актобе,0.105263,0.106599
Алматы,0.080537,0.074557
Астана,0.093233,0.083086
Атырау,0.086957,0.067568
Караганда,0.087591,0.093878
Шымкент,0.089136,0.306954


In [51]:
df.groupby(["city", "month"])["cancel"].mean().unstack().round(3)   # месяцы → в столбцы

month,9,10
city,,
Актобе,0.105,0.107
Алматы,0.081,0.075
Астана,0.093,0.083
Атырау,0.087,0.068
Караганда,0.088,0.094
Шымкент,0.089,0.307


In [52]:
df.groupby(["month","city"])["cancel"].mean().unstack()

city,Актобе,Алматы,Астана,Атырау,Караганда,Шымкент
month,,,,,,
9,0.105263,0.080537,0.093233,0.086957,0.087591,0.089136
10,0.106599,0.074557,0.083086,0.067568,0.093878,0.306954


In [53]:
df.groupby(["month","city"])["cancel"].mean().unstack().T

month,9,10
city,,
Актобе,0.105263,0.106599
Алматы,0.080537,0.074557
Астана,0.093233,0.083086
Атырау,0.086957,0.067568
Караганда,0.087591,0.093878
Шымкент,0.089136,0.306954


In [54]:
paid.groupby('city')['order_id'].agg(['mean','median'])

,mean,median
city,,
Актобе,102624.708092,102642.5
Алматы,102657.956967,102639.0
Астана,102662.470925,102635.0
Атырау,102599.303030,102664.0
Караганда,102515.853814,102397.0
Шымкент,102531.951299,102498.0


## Практика 1, 12 минут
1. **[База]** Сколько заказов и какая выручка оплаченных в каждой категории? Отсортируйте по выручке. Сходится с ДЗ 4 (лидер Электроника, 34,4 млн ₸)?
2. **[База]** Таблица по городам: число заказов и доля отмен (`agg` с именами). Какой город выделяется и на скольких заказах это видно?
3. **[Стандарт]** Средний и медианный чек оплаченных заказов по городам (`agg(["mean", "median"])`). Может, Шымкент падает, потому что там покупают дешевле?
4. **[Стандарт]** Доля отмен по сроку доставки: `groupby("delivery_days")`, рядом число заказов. Как отмены зависят от срока?
5. **[Челлендж]** Средний срок доставки по городам и месяцам (`groupby` по двум столбцам + `unstack`). Какая клетка выделяется? Почему `count` и `size` в этой таблице дают разные числа?

# 1[База] Сколько заказов и какая выручка оплаченных в каждой категории? Отсортируйте по выручке. Сходится с ДЗ 4 (лидер Электроника, 34,4 млн ₸)?

In [55]:
by_category = paid.groupby("category").agg(
    orders=("order_id", "count"),
    revenue=("amount", "sum")
)

In [56]:
by_category.sort_values("revenue", ascending=False)

,orders,revenue
category,,
Электроника,1036,34370430.0
Одежда,1330,17672830.0
Дом,952,10123520.0
Красота,857,5289940.0
Книги,564,2206210.0


# 2[База] Таблица по городам: число заказов и доля отмен (agg с именами). Какой город выделяется и на скольких заказах это видно?

In [57]:
by_city = df.groupby("city").agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean")
)

In [58]:
by_city.sort_values("cancel_share", ascending=False).round(3)

,orders,cancel_share
city,,
Шымкент,776,0.206
Актобе,387,0.106
Караганда,519,0.091
Астана,1339,0.088
Алматы,2116,0.078
Атырау,143,0.077


# 3[Стандарт] Средний и медианный чек оплаченных заказов по городам (agg(["mean", "median"])). Может, Шымкент падает, потому что там покупают дешевле?

In [59]:
paid.groupby("city")["amount"].agg(["mean", "median"]).round(2)

,mean,median
city,,
Актобе,14790.12,10210.0
Алматы,14930.37,10445.0
Астана,14647.62,10120.0
Атырау,14347.35,9920.0
Караганда,14873.31,10455.0
Шымкент,13965.36,9995.0


# 4[Стандарт] Доля отмен по сроку доставки: groupby("delivery_days"), рядом число заказов. Как отмены зависят от срока?

In [60]:
df.groupby("delivery_days").agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean")
).round(3)

,orders,cancel_share
delivery_days,,
1.0,1557,0.042
2.0,1570,0.080
3.0,1590,0.134
5.0,86,0.221
6.0,120,0.217
7.0,95,0.347
8.0,104,0.433


# 5[Челлендж] Средний срок доставки по городам и месяцам (groupby по двум столбцам + unstack). Какая клетка выделяется? Почему count и size в этой таблице дают разные числа?

In [61]:
df.groupby(["city", "month"])["delivery_days"].mean().unstack().round(2)

month,9,10
city,,
Актобе,2.04,2.10
Алматы,1.98,2.01
Астана,2.05,1.99
Атырау,2.06,2.01
Караганда,1.90,1.99
Шымкент,2.03,6.54


## 2. pivot_table: таблица «город × месяц»
На уроке 4 таблицу 6×2 мы собирали циклом и NumPy. `pivot_table` делает её одной строкой: что в строках (`index`), что в столбцах (`columns`), что считаем (`values`) и как (`aggfunc`).

In [62]:
paid.groupby(['city','month'])['amount'].sum().unstack()

month,9,10
city,,
Актобе,2576960.0,2540420.0
Алматы,14501070.0,14643020.0
Астана,9034220.0,8850530.0
Атырау,923990.0,969860.0
Караганда,3648520.0,3371680.0
Шымкент,4558650.0,4044010.0


In [63]:
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum")
p

month,9,10
city,,
Актобе,2576960.0,2540420.0
Алматы,14501070.0,14643020.0
Астана,9034220.0,8850530.0
Атырау,923990.0,969860.0
Караганда,3648520.0,3371680.0
Шымкент,4558650.0,4044010.0


In [64]:
# ПРЕДСКАЖИТЕ: что посчитает pivot_table, если не указать aggfunc?
paid.pivot_table(index="city", columns="month", values="amount").round()

month,9,10
city,,
Актобе,15159.0,14434.0
Алматы,15121.0,14746.0
Астана,14982.0,14321.0
Атырау,14667.0,14056.0
Караганда,14594.0,15188.0
Шымкент,13941.0,13993.0


> **Частая ошибка в коде от AI:** `pivot_table` без `aggfunc`. По умолчанию это **среднее**, а не сумма: вместо выручки получается средний чек. Код не падает, таблица выглядит правдоподобно.

Добавляем столбцы, как к обычной таблице: изменение и вклад города в падение (урок 4).

In [65]:
p["change_%"] = (p[10] / p[9] - 1) * 100
p["diff"] = p[10] - p[9]
p["share_of_drop_%"] = p["diff"] / p["diff"].sum() * 100
p.sort_values("diff").round(1)

month,9,10,change_%,diff,share_of_drop_%
city,,,,,
Шымкент,4558650.0,4044010.0,-11.3,-514640.0,62.5
Караганда,3648520.0,3371680.0,-7.6,-276840.0,33.6
Астана,9034220.0,8850530.0,-2.0,-183690.0,22.3
Актобе,2576960.0,2540420.0,-1.4,-36540.0,4.4
Атырау,923990.0,969860.0,5.0,45870.0,-5.6
Алматы,14501070.0,14643020.0,1.0,141950.0,-17.2


In [66]:
# margins=True добавляет строку и столбец «All»: итог по компании
paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum", margins=True)

month,9,10,All
city,,,
Актобе,2576960.0,2540420.0,5117380.0
Алматы,14501070.0,14643020.0,29144090.0
Астана,9034220.0,8850530.0,17884750.0
Атырау,923990.0,969860.0,1893850.0
Караганда,3648520.0,3371680.0,7020200.0
Шымкент,4558650.0,4044010.0,8602660.0
All,35243410.0,34419520.0,69662930.0


> Итог тот же, что на уроке 4: −2,3%, и 62,5% падения даёт Шымкент. Теперь вопрос: **что именно** изменилось в Шымкенте?

### Выручка = число оплаченных заказов × средний чек
Разложим выручку на части и посмотрим каждую по городам и месяцам.

In [67]:
df.pivot_table(index="city", columns="month", values="order_id", aggfunc="count")    # заказов

month,9,10
city,,
Актобе,190,197
Алматы,1043,1073
Астана,665,674
Атырау,69,74
Караганда,274,245
Шымкент,359,417


In [68]:
df.pivot_table(index="city", columns="month", values="cancel", aggfunc="mean").round(3)   # доля отмен

month,9,10
city,,
Актобе,0.105,0.107
Алматы,0.081,0.075
Астана,0.093,0.083
Атырау,0.087,0.068
Караганда,0.088,0.094
Шымкент,0.089,0.307


In [69]:
paid.pivot_table(index="city", columns="month", values="amount", aggfunc="mean").round()  # средний чек

month,9,10
city,,
Актобе,15159.0,14434.0
Алматы,15121.0,14746.0
Астана,14982.0,14321.0
Атырау,14667.0,14056.0
Караганда,14594.0,15188.0
Шымкент,13941.0,13993.0


В Шымкенте в октябре заказов **больше** (359 → 417), средний чек почти тот же, а доля отмен выросла с 8,9% до 30,7%. Выручку съели отмены. Отмены растут, когда доставка медленная (Практика 1, пункт 4). Смотрим сроки.

In [70]:
df.pivot_table(index="city", columns="month", values="delivery_days", aggfunc="mean").round(2)

month,9,10
city,,
Актобе,2.04,2.10
Алматы,1.98,2.01
Астана,2.05,1.99
Атырау,2.06,2.01
Караганда,1.90,1.99
Шымкент,2.03,6.54


In [71]:
# Шымкент по дням: groupby по дате (.dt.date – дата без времени)
sh = df[df["city"] == "Шымкент"]
by_day = sh.groupby(sh["order_date"].dt.date)["delivery_days"].mean()
by_day.iloc[26:34].round(2)       # 27.09 – 04.10

order_date
2026-09-27    2.17
2026-09-28    2.19
2026-09-29    2.11
2026-09-30    2.33
2026-10-01    6.21
2026-10-02    7.30
2026-10-03    6.65
2026-10-04    6.36
Name: delivery_days, dtype: float64

## Практика 2, 20 минут
1. **[База]** Таблица «город × месяц»: выручка оплаченных, изменение в % и вклад в падение. Совпадает с уроком 4?
2. **[База]** Таблица «категория × месяц»: выручка оплаченных и изменение в %. Какая категория падает сильнее всех? Сравните с ДЗ 4.
3. **[Стандарт]** Шымкент против остальных городов: число заказов, доля отмен и средний чек по месяцам. Подсказка: `df["is_sh"] = df["city"] == "Шымкент"` и `groupby(["is_sh", "month"])`. Что изменилось в октябре: заказы, отмены или чек?
4. **[Стандарт]** Средний срок доставки «город × месяц» через `pivot_table`. Какая клетка выделяется? Напишите вывод для менеджера одной фразой.
5. **[Челлендж]** Шымкент по дням: средний срок доставки (`groupby` по `.dt.date`). Найдите первый день, когда средний срок не меньше 5. Проверьте, что в остальных городах в этот день срок обычный.

# 1[База] Таблица «город × месяц»: выручка оплаченных, изменение в % и вклад в падение. Совпадает с уроком 4?

In [73]:
revenue = paid.groupby(["city", "month"])["amount"].sum().unstack()

In [74]:
revenue

month,9,10
city,,
Актобе,2576960.0,2540420.0
Алматы,14501070.0,14643020.0
Астана,9034220.0,8850530.0
Атырау,923990.0,969860.0
Караганда,3648520.0,3371680.0
Шымкент,4558650.0,4044010.0


In [75]:
change = (revenue[10] / revenue[9] - 1) * 100

In [76]:
change.round(1)

city
Актобе       -1.4
Алматы        1.0
Астана       -2.0
Атырау        5.0
Караганда    -7.6
Шымкент     -11.3
dtype: float64

In [77]:
contribution = (revenue[10] - revenue[9]) / (revenue[10].sum() - revenue[9].sum()) * 100

In [78]:
contribution.round(1)

city
Актобе        4.4
Алматы      -17.2
Астана       22.3
Атырау       -5.6
Караганда    33.6
Шымкент      62.5
dtype: float64

# 2[База] Таблица «категория × месяц»: выручка оплаченных и изменение в %. Какая категория падает сильнее всех? Сравните с ДЗ 4.

In [79]:
category_revenue = paid.groupby(["category", "month"])["amount"].sum().unstack()

In [80]:
category_revenue

month,9,10
category,,
Дом,4997730.0,5125790.0
Книги,1065300.0,1140910.0
Красота,2610950.0,2678990.0
Одежда,9167100.0,8505730.0
Электроника,17402330.0,16968100.0


In [81]:
category_change = (category_revenue[10] / category_revenue[9] - 1) * 100

In [82]:
category_change.round(1)

category
Дом            2.6
Книги          7.1
Красота        2.6
Одежда        -7.2
Электроника   -2.5
dtype: float64

In [83]:
(category_revenue[10] - category_revenue[9]).sort_values()

category
Одежда        -661370.0
Электроника   -434230.0
Красота         68040.0
Книги           75610.0
Дом            128060.0
dtype: float64

# 3[Стандарт] Шымкент против остальных городов: число заказов, доля отмен и средний чек по месяцам. Подсказка: df["is_sh"] = df["city"] == "Шымкент" и groupby(["is_sh", "month"]). Что изменилось в октябре: заказы, отмены или чек?

In [84]:
df["is_sh"] = df["city"] == "Шымкент"

In [85]:
df.groupby(["is_sh", "month"]).agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean"),
    avg_amount=("amount", "mean")
).round(2)

orders  cancel_share  avg_amount
is_sh month                                  
False 9        2241          0.09    14843.49
      10       2263          0.08    14684.75
True  9         359          0.09    13923.84
      10        417          0.31    14262.11

In [89]:
paid = df[df["status"] == "оплачен"].copy()

In [90]:
paid["is_sh"] = paid["city"] == "Шымкент"

In [91]:
paid.groupby(["is_sh", "month"])["amount"].mean().round(2)

is_sh  month
False  9        15004.77
       10       14617.67
True   9        13940.83
       10       13993.11
Name: amount, dtype: float64

# 4[Стандарт] Средний срок доставки «город × месяц» через pivot_table. Какая клетка выделяется? Напишите вывод для менеджера одной фразой.

In [92]:
df.pivot_table(
    index="city",
    columns="month",
    values="delivery_days",
    aggfunc="mean"
).round(2)

month,9,10
city,,
Актобе,2.04,2.10
Алматы,1.98,2.01
Астана,2.05,1.99
Атырау,2.06,2.01
Караганда,1.90,1.99
Шымкент,2.03,6.54


# 5[Челлендж] Шымкент по дням: средний срок доставки (groupby по .dt.date). Найдите первый день, когда средний срок не меньше 5. Проверьте, что в остальных городах в этот день срок обычный.


In [93]:
sh = df[df["city"] == "Шымкент"]

In [94]:
sh_days = sh.groupby(sh["order_date"].dt.date)["delivery_days"].mean()

In [95]:
sh_days.sort_index()

order_date
2026-09-01    1.714286
2026-09-02    2.041667
2026-09-03    1.583333
2026-09-04    1.823529
2026-09-05    2.100000
                ...   
2026-10-27    6.000000
2026-10-28    6.294118
2026-10-29    6.538462
2026-10-30    6.307692
2026-10-31    6.333333
Name: delivery_days, Length: 61, dtype: float64

In [100]:
first_day = sh_days[sh_days >= 5].index[0]

In [101]:
first_day

datetime.date(2026, 10, 1)

In [102]:
other_days.loc[first_day]

np.float64(1.8888888888888888)

## 3. merge: заказы + клиенты (06-Merging)
Маркетолог говорит: «В октябре упала органика, проблема в SEO». Канала привлечения нет в orders.csv: он в customers.csv. `merge` склеивает две таблицы по общему столбцу, как ВПР в Excel.

MERGE, JOIN, CONCAT

In [103]:
orders_t = pd.DataFrame({"order_id": [1, 2, 3, 4],
                         "customer_id": ["C1", "C2", "C1", "TEST"],
                         "amount": [100, 200, 300, 999]})
clients_t = pd.DataFrame({"customer_id": ["C1", "C2", "C3"],
                          "channel": ["реклама", "органика", "рефералы"]})
clients_t

,customer_id,channel
0,C1,реклама
1,C2,органика
2,C3,рефералы


In [104]:
orders_t

,order_id,customer_id,amount
0,1,C1,100
1,2,C2,200
2,3,C1,300
3,4,TEST,999


In [105]:
# ПРЕДСКАЖИТЕ: сколько строк? По умолчанию how="inner"
orders_t.merge(clients_t, on="customer_id")

,order_id,customer_id,amount,channel
0,1,C1,100,реклама
1,2,C2,200,органика
2,3,C1,300,реклама


In [106]:
orders_t.merge(clients_t, on="customer_id", how="left")

,order_id,customer_id,amount,channel
0,1,C1,100,реклама
1,2,C2,200,органика
2,3,C1,300,реклама
3,4,TEST,999,NaN


`inner` оставляет только строки, у которых ключ есть в **обеих** таблицах: заказ TEST пропал молча. `left` сохраняет все строки левой таблицы, недостающее заполняет NaN.

Вторая ловушка: повторяющийся ключ в справочнике размножает строки.

In [107]:
clients_dup = pd.DataFrame({"customer_id": ["C1", "C1", "C2"],
                            "channel": ["реклама", "реклама", "органика"]})
clients_dup

,customer_id,channel
0,C1,реклама
1,C1,реклама
2,C2,органика


In [108]:
orders_t

,order_id,customer_id,amount
0,1,C1,100
1,2,C2,200
2,3,C1,300
3,4,TEST,999


In [109]:
# ПРЕДСКАЖИТЕ: сколько строк и какая сумма amount?
r = orders_t.merge(clients_dup, on="customer_id", how="left")
len(r), r["amount"].sum()

(6, np.int64(1999))

In [110]:
r

,order_id,customer_id,amount,channel
0,1,C1,100,реклама
1,1,C1,100,реклама
2,2,C2,200,органика
3,3,C1,300,реклама
4,3,C1,300,реклама
5,4,TEST,999,NaN


> **Правило merge:** до и после проверяем `len`, ключ справочника уникален (`is_unique`), пропуски после `left` (`isna().sum()`). Иначе выручка тихо меняется.

### orders.csv + customers.csv

In [111]:
cu = pd.read_csv("customers.csv")
cu.shape, cu["customer_id"].is_unique

((2200, 4), True)

In [112]:
cu.head()

,customer_id,channel,age,signup_date
0,C00001,рефералы,31,2026-08-06
1,C00002,реклама,30,2025-07-16
2,C00003,органика,45,2026-05-22
3,C00004,органика,43,2025-10-11
4,C00005,реклама,35,2025-09-02


In [113]:
df.shape

(5280, 11)

In [114]:
m = df.merge(cu, on="customer_id", how="left")
len(df), len(m), m["channel"].isna().sum()

(5280, 5280, np.int64(0))

In [115]:
m.head()

,order_id,order_date,customer_id,city,category,amount,delivery_days,status,month,cancel,is_sh,channel,age,signup_date
0,101674,2026-09-01 08:00:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен,9,False,False,рефералы,35,2025-02-11
1,101214,2026-09-01 08:00:00,C01210,Астана,Красота,3630.0,2.0,оплачен,9,False,False,органика,45,2025-07-18
2,102424,2026-09-01 08:00:00,C01408,Караганда,Дом,9120.0,3.0,оплачен,9,False,False,органика,22,2025-05-05
3,102333,2026-09-01 08:00:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен,9,False,False,органика,30,2026-05-10
4,102564,2026-09-01 09:00:00,C00657,Астана,Дом,11010.0,1.0,оплачен,9,False,False,рефералы,45,2025-12-05


In [116]:
paid_m = m[m["status"] == "оплачен"]
ch = paid_m.pivot_table(index="channel", columns="month", values="amount", aggfunc="sum")


In [117]:
ch

month,9,10
channel,,
органика,12212420.0,11484020.0
реклама,18034980.0,17467600.0
рефералы,4996010.0,5467900.0


In [118]:
ch["change_%"] = (ch[10] / ch[9] - 1) * 100
ch.round(1)

month,9,10,change_%
channel,,,
органика,12212420.0,11484020.0,-6.0
реклама,18034980.0,17467600.0,-3.1
рефералы,4996010.0,5467900.0,9.4


Органика −6,0%. Значит, проблема в SEO? Проверим в Практике 3.

## Практика 3, 15 минут
1. **[База]** Сколько строк до и после `merge`? Сколько заказов без канала? Сколько клиентов из customers.csv не сделали ни одного заказа в сентябре–октябре (`isin`)?
2. **[База]** Выручка оплаченных по каналам и месяцам, изменение в %.
3. **[Стандарт]** Проверьте гипотезу маркетолога: по каналам и месяцам число оплаченных заказов, средний чек и доля отмен. Какая доля заказов каждого канала из Шымкента? Объясняет ли канал падение?
4. **[Челлендж]** Сделайте `merge` по умолчанию (`inner`) с исходным `pd.read_csv("orders.csv")`. Сколько строк пропало, какие и почему код не предупредил? Как это поймать проверкой?

In [119]:
customers = pd.read_csv("customers.csv")

In [120]:
before = len(df)

In [121]:
merged = df.merge(customers, on="customer_id", how="left")

In [122]:
before, len(merged), merged["channel"].isna().sum()

(5280, 5280, np.int64(0))

# 1[База] Сколько строк до и после merge? Сколько заказов без канала? Сколько клиентов из customers.csv не сделали ни одного заказа в сентябре–октябре (isin)?

In [123]:
(~customers["customer_id"].isin(df["customer_id"])).sum()

np.int64(198)

# 2[База] Выручка оплаченных по каналам и месяцам, изменение в %.

In [124]:
paid = merged[merged["status"] == "оплачен"]

In [125]:
channel_revenue = paid.groupby(["channel", "month"])["amount"].sum().unstack()

In [126]:
channel_revenue

month,9,10
channel,,
органика,12212420.0,11484020.0
реклама,18034980.0,17467600.0
рефералы,4996010.0,5467900.0


In [127]:
((channel_revenue[10] / channel_revenue[9] - 1) * 100).round(1)

channel
органика   -6.0
реклама    -3.1
рефералы    9.4
dtype: float64

# 3[Стандарт] Проверьте гипотезу маркетолога: по каналам и месяцам число оплаченных заказов, средний чек и доля отмен. Какая доля заказов каждого канала из Шымкента? Объясняет ли канал падение?

In [128]:
paid.groupby(["channel", "month"]).agg(
    orders=("order_id", "count"),
    avg_amount=("amount", "mean")
).round(2)

orders  avg_amount
channel  month                    
органика 9         807    15133.11
         10        811    14160.32
реклама  9        1213    14868.08
         10       1194    14629.48
рефералы 9         352    14193.21
         10        362    15104.70

In [129]:
merged["is_sh"] = merged["city"] == "Шымкент"

In [130]:
merged.groupby(["channel", "month"]).agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean"),
    sh_share=("is_sh", "mean")
).round(3)

orders  cancel_share  sh_share
channel  month                                
органика 9         886         0.089     0.139
         10        922         0.120     0.156
реклама  9        1334         0.091     0.135
         10       1351         0.116     0.160
рефералы 9         380         0.074     0.147
         10        407         0.111     0.140

# 4[Челлендж] Сделайте merge по умолчанию (inner) с исходным pd.read_csv("orders.csv"). Сколько строк пропало, какие и почему код не предупредил? Как это поймать проверкой?

In [131]:
raw = pd.read_csv("orders.csv")

In [132]:
inner = raw.merge(customers, on="customer_id", how="inner")

In [133]:
len(raw), len(inner), len(raw) - len(inner)

(5463, 5460, 3)

In [134]:
raw[~raw["customer_id"].isin(customers["customer_id"])][
    ["order_id", "customer_id"]
]

,order_id,customer_id
1001,105281,TEST
2268,105282,TEST
2966,105283,TEST


## Итоги

| Вопрос менеджера | Инструмент | Проверка |
|---|---|---|
| «По каждому городу» | `df.groupby("city")["amount"].sum()` | выручка только по оплаченным |
| Несколько показателей сразу | `.agg(orders=("order_id", "count"), ...)` | рядом с долей число заказов |
| Доля отмен | `groupby(...)["cancel"].mean()` | среднее от True/False |
| Таблица «город × месяц» | `pivot_table(index=, columns=, values=, aggfunc="sum")` | `aggfunc` указан явно |
| Добавить данные из другой таблицы | `merge(..., on="customer_id", how="left")` | `len` до и после, ключ уникален, `isna().sum()` |

Падение −2,3% – это Шымкент: заказов там больше, чек тот же, а отмен в октябре 30,7% вместо 8,9%, потому что доставка идёт 5–8 дней. Канал привлечения падение не объясняет.

**ДЗ 7:** ноутбук HW07_pandas_groupby, сдача в папку `hw07` репозитория `dsa-homework`. На уроке 8 нарисуем эти таблицы графиками.